# Preprocessing

In [191]:
# Import necessary libraries
import numpy as np
import pandas as pd 
from collections import Counter     # Use for value counts within a list


In [192]:
# Pull in the datasets
patient_survey = pd.read_csv('Data/HCAHPS-Hospital.csv')
ha_infections = pd.read_csv('Data/Healthcare_Associated_Infections-Hospital.csv')
timely_effective = pd.read_csv('Data/Timely_and_Effective_Care-Hospital.csv')

C:\Users\ahuyn\AppData\Local\Temp\ipykernel_14096\2035690034.py:2: DtypeWarning: Columns (12,14,17,19) have mixed types. Specify dtype option on import or set low_memory=False.
  patient_survey = pd.read_csv('Data/HCAHPS-Hospital.csv')
C:\Users\ahuyn\AppData\Local\Temp\ipykernel_14096\2035690034.py:4: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  timely_effective = pd.read_csv('Data/Timely_and_Effective_Care-Hospital.csv')


In [193]:
# Assess datasets
datasets = [patient_survey, ha_infections, timely_effective]
all_columns = []

# Create a for loop for details on each dataset
for dataset in datasets:
    if dataset is patient_survey:
        print("Patient Survey", '\n', '--'*40)
        print(dataset.columns)
        print("Dataset Shape: ", dataset.shape)
        for column_name in dataset.columns:
            all_columns.append(column_name)
    elif dataset is ha_infections:
        print('\n', "Healthcare Acquired Infections", '\n', '--'*40)
        print(dataset.columns)
        print("Dataset Shape: ", dataset.shape)
        for column_name in dataset.columns:
            all_columns.append(column_name)
    else:
        print('\n',"Timely and Effectiveness", '\n', '--'*40)
        print(dataset.columns)
        print("Dataset Shape: ", dataset.shape)
        for column_name in dataset.columns:
            all_columns.append(column_name)

Patient Survey 
 --------------------------------------------------------------------------------
Index(['Facility ID', 'Facility Name', 'Address', 'City/Town', 'State',
       'ZIP Code', 'County/Parish', 'Telephone Number', 'HCAHPS Measure ID',
       'HCAHPS Question', 'HCAHPS Answer Description',
       'Patient Survey Star Rating', 'Patient Survey Star Rating Footnote',
       'HCAHPS Answer Percent', 'HCAHPS Answer Percent Footnote',
       'HCAHPS Linear Mean Value', 'Number of Completed Surveys',
       'Number of Completed Surveys Footnote', 'Survey Response Rate Percent',
       'Survey Response Rate Percent Footnote', 'Start Date', 'End Date'],
      dtype='object')
Dataset Shape:  (325856, 22)

 Healthcare Acquired Infections 
 --------------------------------------------------------------------------------
Index(['Facility ID', 'Facility Name', 'Address', 'City/Town', 'State',
       'ZIP Code', 'County/Parish', 'Telephone Number', 'Measure ID',
       'Measure Name', 'Com

In [194]:
# Determine key column candidates from the datasets
column_counts = Counter(all_columns)
print(column_counts)

# Use Facility ID as the key column

Counter({'Facility ID': 3, 'Facility Name': 3, 'Address': 3, 'City/Town': 3, 'State': 3, 'ZIP Code': 3, 'County/Parish': 3, 'Telephone Number': 3, 'Start Date': 3, 'End Date': 3, 'Measure ID': 2, 'Measure Name': 2, 'Score': 2, 'Footnote': 2, 'HCAHPS Measure ID': 1, 'HCAHPS Question': 1, 'HCAHPS Answer Description': 1, 'Patient Survey Star Rating': 1, 'Patient Survey Star Rating Footnote': 1, 'HCAHPS Answer Percent': 1, 'HCAHPS Answer Percent Footnote': 1, 'HCAHPS Linear Mean Value': 1, 'Number of Completed Surveys': 1, 'Number of Completed Surveys Footnote': 1, 'Survey Response Rate Percent': 1, 'Survey Response Rate Percent Footnote': 1, 'Compared to National': 1, 'Condition': 1, 'Sample': 1})


In [195]:
for name, df in {
    "Patient Survey": patient_survey,
    "Timely and Effective Care": timely_effective,
    "Healthcare Acquired Infection": ha_infections}.items():

    print("\n", name)
    print("Rows:", len(df))
    print("Unique Facilities:", df['Facility ID'].nunique())
    print(df['Facility ID'].value_counts().describe())


 Patient Survey
Rows: 325856
Unique Facilities: 4792
count    4792.0
mean       68.0
std         0.0
min        68.0
25%        68.0
50%        68.0
75%        68.0
max        68.0
Name: count, dtype: float64

 Timely and Effective Care
Rows: 138173
Unique Facilities: 4660
count    4660.000000
mean       29.650858
std         1.482888
min        23.000000
25%        30.000000
50%        30.000000
75%        30.000000
max        30.000000
Name: count, dtype: float64

 Healthcare Acquired Infection
Rows: 172512
Unique Facilities: 4792
count    4792.0
mean       36.0
std         0.0
min        36.0
25%        36.0
50%        36.0
75%        36.0
max        36.0
Name: count, dtype: float64


In [196]:
# Assess why there are multiple rows for each unique identifier
for dataset in datasets:

    print('--'*40)

    # Define an example facility -- use the first one
    facility_example = dataset['Facility ID'].iloc[0]

    # Filter the dataset to the example facility only
    subset = dataset[dataset['Facility ID'] == facility_example]

    # Create a for loop to see which columns have unique values when facility ID is the same
    for column in subset.columns:
        if subset[column].nunique() > 1:
            print(column, " Unique Values: ", subset[column].nunique())


--------------------------------------------------------------------------------
HCAHPS Measure ID  Unique Values:  68
HCAHPS Question  Unique Values:  68
HCAHPS Answer Description  Unique Values:  68
Patient Survey Star Rating  Unique Values:  4
HCAHPS Answer Percent  Unique Values:  35
HCAHPS Linear Mean Value  Unique Values:  9
--------------------------------------------------------------------------------
Measure ID  Unique Values:  36
Measure Name  Unique Values:  36
Compared to National  Unique Values:  3
Score  Unique Values:  34
--------------------------------------------------------------------------------
Condition  Unique Values:  6
Measure ID  Unique Values:  30
Measure Name  Unique Values:  30
Score  Unique Values:  16
Sample  Unique Values:  14
Footnote  Unique Values:  3
Start Date  Unique Values:  3
End Date  Unique Values:  3


### Cleaning: HCAHPS Patient Survey

In [197]:
# Get more information on the patient_survey table
patient_survey.info()

# Drop the footnote columns since it seems to be optional
patient_survey = patient_survey.drop(columns = ['Patient Survey Star Rating Footnote', 'HCAHPS Answer Percent Footnote', 'Number of Completed Surveys Footnote', 'Survey Response Rate Percent Footnote'])

# Double check the columns were successfully droppec
print("New Patient Survey Shape: ", patient_survey.shape)

# Since Facility ID will serve as our primary key, the other identifiers will be dropped
survey_df = patient_survey[['Facility ID', 'Facility Name', 'HCAHPS Measure ID', 'HCAHPS Question', 'HCAHPS Answer Description', 'Patient Survey Star Rating', 'Start Date', 'End Date']]

# Ensure that the primary key is a text data type
survey_df["Facility ID"] = survey_df["Facility ID"].astype(str)

# Double check the data types
print(survey_df.info())
print(survey_df.head()) 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 325856 entries, 0 to 325855
Data columns (total 22 columns):
 #   Column                                 Non-Null Count   Dtype 
---  ------                                 --------------   ----- 
 0   Facility ID                            325856 non-null  object
 1   Facility Name                          325856 non-null  object
 2   Address                                325856 non-null  object
 3   City/Town                              325856 non-null  object
 4   State                                  325856 non-null  object
 5   ZIP Code                               325856 non-null  int64 
 6   County/Parish                          325856 non-null  object
 7   Telephone Number                       325856 non-null  object
 8   HCAHPS Measure ID                      325856 non-null  object
 9   HCAHPS Question                        325856 non-null  object
 10  HCAHPS Answer Description              325856 non-null  object
 11  

C:\Users\ahuyn\AppData\Local\Temp\ipykernel_14096\408726337.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  survey_df["Facility ID"] = survey_df["Facility ID"].astype(str)


In [198]:
# Assess the unique values for survey start ratings
print(survey_df['Patient Survey Star Rating'].value_counts())

# Drop the 'Not Applicable' and the 'Not Available' from the dataset
survey_df = survey_df[~survey_df['Patient Survey Star Rating'].isin(['Not Applicable', 'Not Available'])]

# Double check that the appropriate rows were removed
print('\nAfter Dropping the N/A Rows: ')
print('--'*30)
print(survey_df['Patient Survey Star Rating'].value_counts())

Patient Survey Star Rating
Not Applicable    282728
Not Available      14544
3                  10033
4                   8697
2                   5508
5                   3296
1                   1050
Name: count, dtype: int64

After Dropping the N/A Rows: 
------------------------------------------------------------
Patient Survey Star Rating
3    10033
4     8697
2     5508
5     3296
1     1050
Name: count, dtype: int64


In [199]:
# Assess which measures are left after removal of N/A values
print(survey_df[['HCAHPS Measure ID', 'HCAHPS Question']].value_counts())
print("--"*30)

# Turn the Star Ratings into an integer
survey_df['Patient Survey Star Rating'] = survey_df['Patient Survey Star Rating'].astype(int)
# Assess the data types from the survey_df 
print(survey_df.info())
print("--"*30)

# Double check to see the Facility ID data types
numeric_check = pd.to_numeric(survey_df['Facility ID'], errors='coerce').notna().all()
print("All Facility IDs Are Numeric: ", numeric_check)


# Which values are non-numeric
nonnumerics = survey_df.loc[pd.to_numeric(survey_df['Facility ID'], errors='coerce').isna(), 'Facility ID'].unique()
print("Nonnumerical Facility IDs: ")
print(nonnumerics)
print("--"*30)

# Drop the original index
survey_df = survey_df.reset_index(drop = True)
# print(survey_df.head())

HCAHPS Measure ID         HCAHPS Question                            
H_CLEAN_STAR_RATING       Cleanliness - star rating                      3176
H_COMP_1_STAR_RATING      Nurse communication - star rating              3176
H_COMP_2_STAR_RATING      Doctor communication - star rating             3176
H_COMP_5_STAR_RATING      Communication about medicines - star rating    3176
H_COMP_6_STAR_RATING      Discharge information - star rating            3176
H_HSP_RATING_STAR_RATING  Overall hospital rating - star rating          3176
H_QUIET_STAR_RATING       Quietness - star rating                        3176
H_RECMND_STAR_RATING      Recommend hospital - star rating               3176
H_STAR_RATING             Summary star rating                            3176
Name: count, dtype: int64
------------------------------------------------------------
<class 'pandas.core.frame.DataFrame'>
Index: 28584 entries, 4 to 324291
Data columns (total 8 columns):
 #   Column                      Non-

### CLeaning: Healthcare Acquired Infections

In [200]:
# Pull the columns from the ha_infections dataset
print(ha_infections.columns)

# Remove irrelevant columns and details
ha_infections = ha_infections[['Facility ID', 'Facility Name', 'Measure ID', 'Measure Name', 'Compared to National', 'Score', 'Start Date', 'End Date']]

# Double check the ha_infection dataset
print(ha_infections.head(5))

Index(['Facility ID', 'Facility Name', 'Address', 'City/Town', 'State',
       'ZIP Code', 'County/Parish', 'Telephone Number', 'Measure ID',
       'Measure Name', 'Compared to National', 'Score', 'Footnote',
       'Start Date', 'End Date'],
      dtype='object')
  Facility ID                    Facility Name       Measure ID  \
0      010001  SOUTHEAST HEALTH MEDICAL CENTER    HAI_1_CILOWER   
1      010001  SOUTHEAST HEALTH MEDICAL CENTER    HAI_1_CIUPPER   
2      010001  SOUTHEAST HEALTH MEDICAL CENTER       HAI_1_DOPC   
3      010001  SOUTHEAST HEALTH MEDICAL CENTER  HAI_1_ELIGCASES   
4      010001  SOUTHEAST HEALTH MEDICAL CENTER  HAI_1_NUMERATOR   

                                        Measure Name  \
0  Central Line Associated Bloodstream Infection ...   
1  Central Line Associated Bloodstream Infection ...   
2  Central Line Associated Bloodstream Infection:...   
3  Central Line Associated Bloodstream Infection ...   
4  Central Line Associated Bloodstream Infection ..

In [201]:
# Find the unique values for Measure Names
print(ha_infections['Measure Name'].value_counts())
# print(ha_infections['Measure Name'].nunique())

# Remove the columns that contain "Confident Limit"
ha_df = ha_infections[~ha_infections['Measure Name'].str.contains('Confidence Limit')]
ha_df = ha_df[~ha_df['Measure Name'].str.contains('Predicted')].reset_index(drop = True)

# Double check the columns are properly dropped
print(ha_df['Measure Name'].value_counts())
#print(ha_df['Measure Name'].nunique())

Measure Name
Central Line Associated Bloodstream Infection (ICU + select Wards): Lower Confidence Limit            4792
Central Line Associated Bloodstream Infection (ICU + select Wards): Upper Confidence Limit            4792
SSI - Abdominal Hysterectomy: Number of Procedures                                                    4792
SSI - Abdominal Hysterectomy: Predicted Cases                                                         4792
SSI - Abdominal Hysterectomy: Observed Cases                                                          4792
SSI - Abdominal Hysterectomy                                                                          4792
MRSA Bacteremia: Lower Confidence Limit                                                               4792
MRSA Bacteremia: Upper Confidence Limit                                                               4792
MRSA Bacteremia: Patient Days                                                                         4792
MRSA Bacteremia: Predict

In [202]:
# Remove the scores that have N/A values
ha_df = ha_df[~ha_df['Score'].isin(['Not Available', 'Not Applicable'])]

# Double check what the size is now
print("Original Shape: ", ha_infections.shape)
print("New Shape: ", ha_df.shape)

print(ha_df['Measure Name'].unique())

Original Shape:  (172512, 8)
New Shape:  (54327, 8)
['Central Line Associated Bloodstream Infection: Number of Device Days'
 'Central Line Associated Bloodstream Infection (ICU + select Wards): Observed Cases'
 'Central Line Associated Bloodstream Infection (ICU + select Wards)'
 'Catheter Associated Urinary Tract Infections (ICU + select Wards): Number of Urinary Catheter Days'
 'Catheter Associated Urinary Tract Infections (ICU + select Wards): Observed Cases'
 'Catheter Associated Urinary Tract Infections (ICU + select Wards)'
 'SSI - Colon Surgery: Number of Procedures'
 'SSI - Colon Surgery: Observed Cases' 'SSI - Colon Surgery'
 'SSI - Abdominal Hysterectomy: Number of Procedures'
 'SSI - Abdominal Hysterectomy: Observed Cases'
 'MRSA Bacteremia: Patient Days' 'MRSA Bacteremia: Observed Cases'
 'MRSA Bacteremia' 'Clostridium Difficile (C.Diff): Patient Days'
 'Clostridium Difficile (C.Diff): Observed Cases'
 'Clostridium Difficile (C.Diff)' 'SSI - Abdominal Hysterectomy']


In [203]:
# Check the scores valid
ha_df['Score'] = ha_df['Score'].astype(float)

# Double check ha_df
print(ha_df.head(5))

  Facility ID                    Facility Name       Measure ID  \
0      010001  SOUTHEAST HEALTH MEDICAL CENTER       HAI_1_DOPC   
1      010001  SOUTHEAST HEALTH MEDICAL CENTER  HAI_1_NUMERATOR   
2      010001  SOUTHEAST HEALTH MEDICAL CENTER        HAI_1_SIR   
3      010001  SOUTHEAST HEALTH MEDICAL CENTER       HAI_2_DOPC   
4      010001  SOUTHEAST HEALTH MEDICAL CENTER  HAI_2_NUMERATOR   

                                        Measure Name  \
0  Central Line Associated Bloodstream Infection:...   
1  Central Line Associated Bloodstream Infection ...   
2  Central Line Associated Bloodstream Infection ...   
3  Catheter Associated Urinary Tract Infections (...   
4  Catheter Associated Urinary Tract Infections (...   

                 Compared to National      Score  Start Date    End Date  
0  Better than the National Benchmark   9871.000  07/01/2024  06/30/2025  
1  Better than the National Benchmark      3.000  07/01/2024  06/30/2025  
2  Better than the National Benchma

### Cleaning: Timely and Effective Care

In [204]:
# Pull the column names
# print(timely_effective.columns)

# Remove the irrelevant details
timely_df = timely_effective[['Facility ID', 'Facility Name', 'State',
                                     'Condition', 'Measure ID', 'Measure Name', 
                                     'Score','Start Date', 'End Date']]

# Double check the columns were successfully dropped
# print(timely_df['Score'].unique())

timely_df = timely_df[timely_effective['Score'] != 'Not Available']

# Double check
print(timely_df.head(5))

   Facility ID                    Facility Name State  \
0       010001  SOUTHEAST HEALTH MEDICAL CENTER    AL   
9       010001  SOUTHEAST HEALTH MEDICAL CENTER    AL   
10      010001  SOUTHEAST HEALTH MEDICAL CENTER    AL   
11      010001  SOUTHEAST HEALTH MEDICAL CENTER    AL   
14      010001  SOUTHEAST HEALTH MEDICAL CENTER    AL   

                           Condition Measure ID  \
0               Emergency Department        EDV   
9   Healthcare Personnel Vaccination      IMM_3   
10              Emergency Department     OP_18a   
11              Emergency Department     OP_18b   
14              Emergency Department      OP_22   

                                         Measure Name      Score  Start Date  \
0                         Emergency department volume  very high  01/01/2024   
9      Healthcare workers given influenza vaccination         93  10/01/2024   
10  Average (median) time all patients spent in th...        218  07/01/2024   
11  Average (median) time pati

In [205]:
# Assess the measures in timely_effective table
print(timely_df[['Measure ID', 'Measure Name']].value_counts().sort_index())


Measure ID           Measure Name                                                                                                                                                                                                                               
EDV                  Emergency department volume                                                                                                                                                                                                                    3837
HH_HYPER             Hospital Harm - Severe Hyperglycemia                                                                                                                                                                                                            939
HH_HYPO              Hospital Harm - Severe Hypoglycemia                                                                                                                                                             

In [206]:
print(survey_df.columns)
print(ha_df.columns)
print(timely_df.columns)

Index(['Facility ID', 'Facility Name', 'HCAHPS Measure ID', 'HCAHPS Question',
       'HCAHPS Answer Description', 'Patient Survey Star Rating', 'Start Date',
       'End Date'],
      dtype='object')
Index(['Facility ID', 'Facility Name', 'Measure ID', 'Measure Name',
       'Compared to National', 'Score', 'Start Date', 'End Date'],
      dtype='object')
Index(['Facility ID', 'Facility Name', 'State', 'Condition', 'Measure ID',
       'Measure Name', 'Score', 'Start Date', 'End Date'],
      dtype='object')


### Save the cleaned datasets

In [207]:
# Save the datasets as csv files for later use
survey_df.to_csv("data/survey_df_clean.csv", index=False)
ha_df.to_csv("data/ha_df_clean.csv", index=False)
timely_df.to_csv("data/timely_df_clean.csv", index=False)